# Centralized Governance & Security Pipeline - On-Demand

Generic metadata-driven notebook to enforce Unity Catalog security, role-based access control (RBAC), and table/view grants centrally from `configs/governance/access_control.json`.

This pipeline runs **on-demand** (or as a scheduled security audit workflow task) and is decoupled from daily ETL runs.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/configs", "Config Directory")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "staging", "prod"], "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.text("target_catalog", "", "Target Catalog (Optional Filter)")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"], "Dry Run (Audit Only)")

config_dir = dbutils.widgets.get("config_dir")
try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
target_catalog = dbutils.widgets.get("target_catalog").strip() or None
dry_run = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Executing Governance Security Pipeline:")
print(f"  - Environment:    {environment}")
print(f"  - Client Scope:   {client_id or 'Global (All Clients)'}")
print(f"  - Catalog Filter: {target_catalog or 'All Catalogs'}")
print(f"  - Dry Run Mode:   {dry_run}")

In [ ]:
# Imports & Path setup
import sys
sys.path.append("/Workspace/Users/jayarampogakula@gmail.com/lakeforge")

from lakeforge.governance import create_governance_engine, GovernanceEngine
print("✅ LakeForge Governance Engine imported successfully")

In [ ]:
# Load centralized governance configuration
gov_config = GovernanceEngine.load_governance_config(config_dir, client_id=client_id)
engine = create_governance_engine(spark=spark)
ddl_statements = engine.generate_grant_ddl(gov_config, target_catalog=target_catalog)

print(f"🔑 Parsed {len(gov_config.roles)} active roles")
print(f"📜 Generated {len(ddl_statements)} Unity Catalog GRANT statements:")
for idx, stmt in enumerate(ddl_statements, 1):
    print(f"  {idx:02d}. {stmt}")

In [ ]:
# Execute or preview governance reconciliation
result = engine.reconcile_permissions(
    config_path_or_dir=config_dir,
    environment=environment,
    dry_run=dry_run,
    target_catalog=target_catalog,
    client_id=client_id
)

print("=" * 60)
print("🛡️ GOVERNANCE RECONCILIATION SUMMARY")
print("=" * 60)
print(f"Total Grants:  {result['total_statements']}")
print(f"Executed:      {result['executed']}")
print(f"Failed:        {result['failed']}")

if result["errors"]:
    print("
❌ Errors encountered during reconciliation:")
    for err in result["errors"]:
        print(f"  * {err}")
else:
    print("
✅ All Unity Catalog permissions reconciled successfully!")
print("=" * 60)